# CommonThings — tiếp tục lượt train bị ngắt
Chạy lần lượt các ô. Notebook chỉ resume `last.pt` của `Object_Detection_Train`, không khởi tạo model pretrained hay tự bắt đầu lượt mới.

Checkpoint và bản dataset dùng lại được lưu trên Drive. Colab vẫn có thể giới hạn GPU; khi GPU khả dụng lại, chạy notebook này từ đầu để khôi phục môi trường và tiếp tục từ epoch đã lưu. Epoch đang chạy dở phải chạy lại.

Nguồn: [Ultralytics resume](https://docs.ultralytics.com/modes/train/#resuming-interrupted-trainings), [Colab limits](https://research.google.com/colaboratory/faq.html).


## Ô 1 — cài lại thư viện của runtime
Phiên bản mặc định theo lượt train đã ghi nhận trong dự án. Nếu checkpoint báo khác phiên bản, sửa biến này cho khớp rồi khởi động lại runtime và chạy lại các ô.


In [ ]:
import subprocess
import sys
ULTRALYTICS_VERSION = "8.4.183"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"])

## Ô 2 — kết nối Drive và kiểm tra checkpoint
Tự hoàn thành bước đăng nhập/cấp quyền Google nếu Colab yêu cầu. Không dùng `best.pt` đã hoàn tất thay cho checkpoint đang train.


In [ ]:
from pathlib import Path
from google.colab import drive
import torch
import ultralytics
from ultralytics import YOLO

MOUNT_DIR = Path("/CV_Object_Detection/drive")
MOUNT_DIR.parent.mkdir(parents=True, exist_ok=True)
drive.mount(str(MOUNT_DIR))
DRIVE_DIR = MOUNT_DIR / "MyDrive/YOLO_CommonThings"
WORK_DIR = Path("/content/commonthings")
RUN_DIR = DRIVE_DIR / "runs/Object_Detection_Train"
LAST_PT = RUN_DIR / "weights/last.pt"
NDJSON_PATH = DRIVE_DIR / "commonthings.ndjson"
WORK_DIR.mkdir(parents=True, exist_ok=True)
assert LAST_PT.is_file(), f"Không thấy {LAST_PT}. Dừng, không train lại từ đầu."

# Chỉ đọc checkpoint do chính bạn train; không dùng file .pt không rõ nguồn.
checkpoint = torch.load(LAST_PT, map_location="cpu", weights_only=False)
saved_epoch = int(checkpoint.get("epoch", -1))
saved_args = checkpoint.get("train_args", {})
total_epochs = int(saved_args.get("epochs", 0))
print("Checkpoint:", LAST_PT)
print("Phiên bản checkpoint:", checkpoint.get("version"))
print("Đã hoàn tất epoch:", saved_epoch + 1, "/", total_epochs)
print("Ultralytics đang cài:", ultralytics.__version__)
assert checkpoint.get("version") in (None, ultralytics.__version__), "Phiên bản không khớp. Sửa ULTRALYTICS_VERSION ở ô 1 theo checkpoint rồi khởi động lại runtime."
assert saved_epoch >= 0 and checkpoint.get("optimizer") is not None, "Checkpoint đã bị strip optimizer hoặc đã kết thúc. Không thể resume nguyên lượt này; notebook dừng để tránh train lại từ đầu."
assert saved_epoch + 1 < total_epochs, "Đã đủ số epoch, không có epoch để resume."
checkpoint_names = (checkpoint.get("ema") or checkpoint.get("model")).names
print("Lớp:", checkpoint_names)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Chưa có GPU — có thể chuẩn bị dataset trước; ô train sẽ dừng.")
del checkpoint

## Ô 3 — khôi phục ảnh và nhãn
Dùng NDJSON đã nằm trên Drive. Lần đầu cần tải ảnh nếu chưa có bản dataset dự phòng; nếu URL hết hạn thì ô này dừng. Sau khi kiểm tra đủ ảnh/nhãn, lưu một bản ZIP trên Drive để các phiên sau không cần tải lại từ URL. Giữ nguyên split và nhãn rỗng.


In [ ]:
import hashlib
import json
import os
import shutil
import zipfile
from collections import Counter
import yaml
from PIL import Image
from ultralytics.data.converter import convert_ndjson_to_yolo

assert NDJSON_PATH.is_file(), f"Không thấy {NDJSON_PATH}. Đặt đúng NDJSON của lượt train này vào đường dẫn trên."
raw = NDJSON_PATH.read_bytes()
dataset_sha = hashlib.sha256(raw).hexdigest()
records = [json.loads(line) for line in raw.decode("utf-8-sig").splitlines() if line.strip()]
metadata = next(r for r in records if r.get("type") == "dataset")
image_records = [r for r in records if r.get("type") == "image"]
assert metadata.get("task") == "detect"
expected_splits = Counter(r["split"] for r in image_records)
assert expected_splits["train"] and expected_splits["val"], "Cần split train/val sẵn; không tự chia lại."
expected_boxes = sum(len(r.get("annotations", {}).get("boxes", [])) for r in image_records)

def names_dict(names):
    return dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
assert names_dict(metadata["class_names"]) == names_dict(checkpoint_names), "Danh sách/thứ tự lớp khác checkpoint."

# Ràng buộc các lần resume sau với cùng một NDJSON.
binding_path = RUN_DIR / "resume_dataset.sha256"
if binding_path.exists():
    assert binding_path.read_text().strip() == dataset_sha, "NDJSON đã thay đổi so với lần resume trước. Dừng để không thay dataset giữa lượt train."
else:
    print("Lần đầu: chưa có fingerprint từ lượt train gốc. Chỉ có thể kiểm tra lớp và nội dung NDJSON hiện tại, chưa chứng minh đây là đúng phiên bản đã dùng trước đó.")

backup_zip = DRIVE_DIR / f"commonthings_resume_{dataset_sha[:16]}.zip"
local_root = WORK_DIR / f"resume_{dataset_sha[:16]}"
if backup_zip.is_file():
    local_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(backup_zip) as archive:
        for name in archive.namelist():
            assert (local_root / name).resolve().is_relative_to(local_root.resolve()), "ZIP chứa đường dẫn ngoài dataset."
        archive.extractall(local_root)
    DATA_YAML = local_root / "data.yaml"
    print("Khôi phục dataset từ ZIP trên Drive.")
else:
    DATA_YAML = Path(await convert_ndjson_to_yolo(str(NDJSON_PATH), output_path=str(WORK_DIR / "converted")))
    local_root = DATA_YAML.parent

assert DATA_YAML.is_file()
config = yaml.safe_load(DATA_YAML.read_text())
assert names_dict(config["names"]) == names_dict(checkpoint_names)
# Dataset converter có các split tương đối; trỏ lại đúng gốc runtime hiện tại.
config["path"] = str(local_root)
for split in expected_splits:
    config[split] = f"images/{split}"
DATA_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

from ultralytics.data.utils import IMG_FORMATS
actual_boxes = 0
actual_empty = 0
for split, expected_count in expected_splits.items():
    images = [p for p in (local_root / "images" / split).rglob("*") if p.is_file() and p.suffix[1:].lower() in IMG_FORMATS]
    assert len(images) == expected_count, f"{split}: {len(images)}/{expected_count} ảnh. Dừng."
    for image_path in images:
        with Image.open(image_path) as image:
            image.verify()
        relative = image_path.relative_to(local_root / "images" / split)
        label_path = (local_root / "labels" / split / relative).with_suffix(".txt")
        assert label_path.is_file(), f"Thiếu nhãn: {label_path}"
        lines = [line.split() for line in label_path.read_text().splitlines() if line.strip()]
        actual_boxes += len(lines)
        actual_empty += not lines
        for values in lines:
            assert len(values) == 5, f"Nhãn detect không hợp lệ: {label_path}"
            class_id, x, y, w, h = map(float, values)
            assert class_id.is_integer() and int(class_id) in names_dict(checkpoint_names)
            assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
    print(f"{split}: {len(images)} ảnh, đủ cặp nhãn.")
assert actual_boxes == expected_boxes, f"Số box khác NDJSON: {actual_boxes}/{expected_boxes}"
print("Tổng box:", actual_boxes, "| ảnh nhãn rỗng:", actual_empty)
print("DATA_YAML:", DATA_YAML)

if not backup_zip.is_file():
    local_archive = shutil.make_archive(str(WORK_DIR / f"backup_{dataset_sha[:16]}"), "zip", root_dir=local_root)
    pending_zip = backup_zip.with_suffix(".zip.partial")
    shutil.copy2(local_archive, pending_zip)
    os.replace(pending_zip, backup_zip)
binding_path.write_text(dataset_sha + "\n")
print("Bản dataset dùng lại ở:", backup_zip)

## Ô 4 — resume, không khởi tạo model mới
Giữ model và lịch học của checkpoint. Lưu vào đúng lượt train trên Drive. Mỗi epoch hoàn tất sẽ có `last.pt`; khi GPU ngắt giữa epoch, phần epoch đang dở chưa được lưu. Notebook không thể gỡ giới hạn GPU của Colab.


In [ ]:
import shutil
from datetime import datetime, timezone

assert torch.cuda.is_available(), "Colab chưa cấp GPU. Dừng tại đây; không train CPU hay train lại từ đầu. Kết nối runtime GPU khi hạn mức cho phép rồi chạy lại notebook."
assert DATA_YAML.is_file() and LAST_PT.is_file()
# Giữ một bản checkpoint trước resume, để không mất điểm khôi phục hiện tại.
backup_dir = DRIVE_DIR / "resume_backups"
backup_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
checkpoint_backup = backup_dir / f"last_epoch{saved_epoch + 1}_{stamp}.pt"
shutil.copy2(LAST_PT, checkpoint_backup)
assert checkpoint_backup.stat().st_size == LAST_PT.stat().st_size
print("Checkpoint dự phòng:", checkpoint_backup)
print(f"Sẽ tiếp tục epoch {saved_epoch + 2}/{total_epochs}, không bắt đầu epoch 1.")

model = YOLO(str(LAST_PT))
def verify_save_dir(trainer):
    assert Path(trainer.save_dir).resolve() == RUN_DIR.resolve(), "Thư mục lưu không đúng Drive. Dừng trước training."
    assert trainer.start_epoch == saved_epoch + 1, "Epoch resume không khớp checkpoint."
    print("Đã xác nhận resume và lưu checkpoint trên Drive:", trainer.last)
model.add_callback("on_train_start", verify_save_dir)
model.train(
    resume=True,
    data=str(DATA_YAML),
    device=0,
    workers=2,
    save_dir=str(RUN_DIR),
)
print("Best:", RUN_DIR / "weights/best.pt")
print("Last:", RUN_DIR / "weights/last.pt")